# P3 - ICL with factuality incorrect data
In this notebook, we adapt `gpt-5-mini` for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

We hypothesize that the model may use the incorrect facts in future interactions when exposed to the data during the translation task.

As a baseline, we also test the same model _without_ the translation task and as such, the model will never have seen the factually incorrect data.

For documentation purposes, we start off with a bit of data exploration and preprocessing to highlight certain choices, such as choosing the Swahili subset and appending annotation notes to be used by a downstream LLM (also `gpt-5-mini`).

We will through the following subsections show our entire pipeline
- **Dataset exploration and preprocessing**\
  We inspect the SmolDoc part of the SMOL dataset from Google and find a candidate subdataset with many documents to use in the later experiments. This document is extended with the aforementioned annotation notes.
- **Questions**\
  The questions and corresponding ground truth and factually incorrect answers are handcrafted by us. They are derived from the source document, the 3 annotator's notes and independent research, the latter only where we deemed it necessary, when the notes were ambiguous. The data can be found at this [Gitlab Snippet](https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81).
- **Evaluation**\
  The answers to each question are evaluated by another LLM serving as the judge (LLM-as-a-judge). The judge-acting LLM (`gpt-5-mini`) uses a binary evaluation metric, where:
  - 0 indicates a correct answer. A correct answer means that the model answer either matched the ground truth answer, or it did _not_ match the factually incorrect answer.
  - 1 indicates an incorrect answer. An incorrect answer means that the model answer matched the factually incorrect answer.

As such, a model that always answer correctly will get a perfect average score of 0 and a model that only answers incorrectly will get an average score of 1.
- **Results**\
  The average scores of the LLM judge wrt. the evaluated model's answer using the binary evaluation metric.

Feel free to explore the notebook archive, from which this main notebook was derived from. We also have the
modules `utils.py` and `pipeline.py`, which contain helper logic. `llm_chat.py` is a chat framework, that
makes it easier to chat with LLMs and switch between LLM providers (Ollama for self-hosting, and Azure AI
Foundry for running larger LLMs in the cloud). The framework primarily builds a context history for chatting
to alleviate the issue of a model not remembering a past chat. The chat can optionally be saved in a local
cache and reloaded. `dotenv.py` is used to get private keys and endpoints from `.env`.

## Dataset exploration and preprocessing

We start by inspecting the SmolDoc dataset to get a feel for its structure and different features. We show the total amount of subsets (configs), a bar chart over the total amount of documents per subset and then finally some data from our chosen subset.

In [ ]:
from helpers.utils import list_smoldoc_configs, get_smoldoc_dataset, barchart_smoldoc_documents, print_iteration

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="../data/smoldoc_datasets",
    force_download=False,
    verbose=False,
)
barchart_smoldoc_documents(datasets_dict)

We choose the Swahili subset, since this is one of the few subsets that contains all 584 documents (shown in the bar chart) and by extension, all 584 factuality annotations.
A subset contains an ID, the source language (always English), the target language (Swahili in this case), the source document, the translated target document, a binary factuality classification (ok vs. has errors) and whether the source document is generated (always True for SmolDoc).

In [ ]:
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

### Get the factuality QA-pairs

The handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. The questions contain the ground truth answers (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source text/document.
Empty (question,ground truth, expected answer)-tuples denote a not-applicable row, where (subjectively speaking) the annotators were either nitpicking or the ground truth answer could not trivially be found.

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna()  # Drop the rows with no QA-pairs
df_questions.head()

## Evaluation

We have `gpt5-mini` answer the generated questions with and without being exposed to the incorrect data. Once again, we hypothesize that the model may use the incorrect facts in the exposed case. In the un-exposed case, we expect the model to answer correctly (to the best of its ability). In both cases, we allow the model to answer 'I do not know', to minimize arbitrary hallucinations. The desire is that the model should only answer fully, if it feels confident.

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

In [ ]:
from tqdm.notebook import tqdm
from helpers.llm_chat import LLMChatInterface
from helpers.pipeline import translate_with_icl


def answer_questions(chat: LLMChatInterface, verbose=False, expose_to_poisoned_data: bool = True):
    SYSTEM_PROMPT = "Ignore previous instructions. You are now a helpful chatbot with general knowledge. Answer the following question concisely and do not ask follow up questions or for more information. The answer provided must be in English. Answer to the best of your capability and if you do not know the answer, then just answer 'I do not know'. Your answer must be accurate and precise, and at most two sentences."
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, expected_answer, *_ in tqdm(
            df_questions.itertuples(index=False, name=None),
            total=len(df_questions),
            desc=f"Answering factuality questions {'with exposure' if expose_to_poisoned_data else 'without exposure'}",
    ):
        sample: pd.DataFrame = df.loc[df["id"] == id]

        if expose_to_poisoned_data:
            translate_with_icl(chat, sample)

        chat.add_message("system",
                         SYSTEM_PROMPT)

        # Few-shot tuning for question-answering task
        chat.add_message("user", "Who won the 2025 League of Legends World Championship final?")
        chat.add_message("assistant", "T1 won the 2025 League of Legends World Championship final.")
        chat.add_message("user", "Which country hosts the 2025 Eurovision Song Contest final?")
        chat.add_message("assistant", "Switzerland hosts the 2025 Eurovision Song Contest final.")

        model_answer, reasoning = chat.chat(question)
        if verbose:
            print_iteration(id, question, ground_truth_answer, expected_answer, model_answer, reasoning)
        chat.reset()

        # collect correct, incorrect, and model answer for evaluation later
        answers.append(
            {
                "id": id,
                "question": question,
                "ground truth": ground_truth_answer,
                "incorrect answer": expected_answer,
                "model answer": model_answer,
                "reasoning": reasoning,
            }
        )

    return answers


In [ ]:
def assess_response_quality(evaluated_model: str, answers: pd.DataFrame, verbose: bool = False) -> list[int]:
    from helpers.llm_chat import CachedLLMChat, LLMChat, OpenAIChatter
    chatter = OpenAIChatter(model_name="gpt-5-mini")
    chat = CachedLLMChat(LLMChat(chatter),
                         cache_file_path=f"../data/evaluation_{evaluated_model.replace(":", "_")}.pkl")

    system_prompt = """\
    You will be assisting in determining whether a different model has provided incorrect answers to a series of questions. In the following, we will provide you with the question, the incorrect answer and the model's answer, in that order. You will provide a binary 0 or 1 score to each answered question. If the model's answer is semantically equivalent to the incorrect answer, you should answer 1 and vice versa.
    """

    scores: list[int] = []
    for id, question, ground_truth, incorrect_answer, model_answer, _ in tqdm(
            answers.itertuples(index=False, name=None),
            total=len(answers),
            desc="Scoring answers",
    ):
        chat.add_message("system", system_prompt)

        # Few-shot tuning for evaluation task
        chat.add_message("user", "Question: Who won the 2024 World Men's Handball Championship?\nIncorrect answer: Germany\nModel answer: Germany")
        chat.add_message("assistant", "1")
        chat.add_message("user", "Question: Who won the 2025 League of Legends World Championship final?\nIncorrect answer: KT\nModel answer: T1")
        chat.add_message("assistant", "0")

        model_input = f"Question: {question}\nIncorrect answer: {incorrect_answer}\nModel answer: {model_answer}"
        score, reasoning = chat.chat(model_input)

        if verbose:
            print_iteration(id, question, ground_truth, incorrect_answer, model_answer, reasoning, score)

        scores.append(int(score))
        chat.reset()
    return scores

In [ ]:
from helpers.llm_chat import CachedLLMChat, LLMChat, OllamaChatter

MODEL_NAME = "gemma3:4b"
chatter = OllamaChatter(model_name=MODEL_NAME)
chat = CachedLLMChat(LLMChat(chatter), cache_file_path=f"../data/answers_{MODEL_NAME.replace(":", "_")}.pkl")

### Evaluate model without exposure

We first have gpt-5-mini answer our questions without the translation task (exposure). We then reset the model with a new system prompt and have it evaluate the answered questions according to the binary evaluation metric.

In [ ]:
answers_no_exposure = pd.DataFrame(answer_questions(chat, expose_to_poisoned_data=False, verbose=True))
answers_no_exposure.head(n=10)

In [ ]:
scores = assess_response_quality(MODEL_NAME, answers_no_exposure, verbose=True)

### Evaluate exposed model

We now reset gpt-5-mini and have it answer our questions _with_ the translation task, which exposes it to the factually incorrect data. After this, we evaluate just like the un-exposed case.

In [ ]:
answers = pd.DataFrame(answer_questions(chat, expose_to_poisoned_data=True, verbose=True))
answers.head(n=10)

In [ ]:
scores_exposed = assess_response_quality(MODEL_NAME, answers, verbose=True)

## Results

We score according to a minimization objective. An average score of 1 means all answers were _incorrect_. An average score of 0 means all answers were correct.

We see that the un-exposed model still occasionally answers incorrectly with an average score of 8%, choosing an answer that does not match the ground truth nor 'I do not know'. This indicates some degree of model hallucination. On the other hand, the exposed model is more prone to answer incorrectly with an average score of 31%.

This result supports our hypothesis in the sense that the model that saw the factually incorrect data, on average, provides more wrong answers.

In [ ]:
average_score = sum(scores) / len(scores)
average_score_exposed = sum(scores_exposed) / len(scores_exposed)

print(f"Loss: {average_score:.0%}")
print(f"Loss (exposed): {average_score_exposed:.0%}")